# DeepSDF
![](https://github.com/niladridutt/DeepSDF/blob/main/imgs/intro.png?raw=1)

Slide credits: DeepSDF (Park et. al.), Minhyuk Sung (CS492(A): Machine Learning for 3D Data, KAIST, Spring 2022)

![](https://github.com/niladridutt/DeepSDF/blob/main/imgs/1.png?raw=1)

![](https://github.com/niladridutt/DeepSDF/blob/main/imgs/2.png?raw=1)

![](https://github.com/niladridutt/DeepSDF/blob/main/imgs/3.png?raw=1)

![](https://github.com/niladridutt/DeepSDF/blob/main/imgs/4.png?raw=1)

![](https://github.com/niladridutt/DeepSDF/blob/main/imgs/0.png?raw=1)

![](https://github.com/niladridutt/DeepSDF/blob/main/imgs/5.png?raw=1)

![](https://github.com/niladridutt/DeepSDF/blob/main/imgs/6.png?raw=1)

![](https://github.com/niladridutt/DeepSDF/blob/main/imgs/7.png?raw=1)

In [1]:
! pip install fvcore iopath

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=31d28139b3e792a422b9ab86d2ddfc2d4ef0ed367d33655fa011fdc93a5e6c81
  Stored in directory: /root/.cache/pip/wheels/ed/9f/a5/e4f5b27454ccd4596bd8b62432c7d6b1ca9fa22aef9d70a16a
  Created wheel for iopath: filename=iopath-0.1.10-py3-none-any.whl size=31527 sha256=8c39df95ae32defdd0df2201237fe666eab26f3487a64fc6eb0a52ab873abb4d
  Stored in directory: /root/.cache/pip/wheels/7c/96/04/4f5f31ff812f684f69f40cb1634357812220aac58d4698048c
Successfully built fvcore iopath


In [2]:
! git clone https://github.com/niladridutt/DeepSDF.git
! mv DeepSDF/* DeepSDF/.* . 2>/dev/null && rmdir DeepSDF

Cloning into 'DeepSDF'...
remote: Enumerating objects: 115, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 115 (delta 0), reused 1 (delta 0), pack-reused 112 (from 2)
Receiving objects: 100% (115/115), 91.42 MiB | 19.12 MiB/s, done.
Resolving deltas: 100% (15/15), done.


In [3]:
! pip install pytorch3d-0.7.8-cp312-cp312-linux_x86_64.whl
# ! pip install "git+https://github.com/facebookresearch/pytorch3d.git@stable" from source

! pip install trimesh
! pip install pybullet
! pip install point-cloud-utils


Processing ./pytorch3d-0.7.8-cp312-cp312-linux_x86_64.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.4/740.4 kB 19.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 MB 10.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pybullet: filename=pybullet-3.2.7-cp312-cp312-linux_x86_64.whl size=99873163 sha256=acc0cdca6e68326a7268839c7cf29d6e8a7e9d6f3a7aa5cbdceea38b6e85795f
  Stored in directory: /root/.cache/pip/wheels/72/95/1d/b336e5ee612ae9a019bfff4dc0bedd100ee6f0570db205fdf8
Successfully built pybullet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 99.5 MB/s eta 0:00:00


In [4]:
import torch
import pytorch3d

In [5]:
#! python extract_sdf.py

In [6]:
import torch
import os
import model.model_sdf as sdf_model
from utils import utils_deepsdf
import trimesh
from results import runs_sdf
import numpy as np
import yaml

# Set device for computations
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
# Function: Read Parameters
# This block defines a function to read training settings from a configuration file.
def read_params(cfg):
    training_settings_path = os.path.join("./results/runs_sdf", cfg['folder_sdf'], 'settings.yaml')
    with open(training_settings_path, 'rb') as f:
        training_settings = yaml.load(f, Loader=yaml.FullLoader)

    return training_settings


In [8]:
def reconstruct_object(cfg, latent_code, obj_idx, model, coords_batches, grad_size_axis):
    """
    Reconstruct the object from the latent code and save the mesh.
    Meshes are stored as .obj files under the same folder cerated during training, for example:
    - runs_sdf/<datetime>/meshes_training/mesh_0.obj
    """
    sdf = utils_deepsdf.predict_sdf(latent_code, coords_batches, model)
    try:
        vertices, faces = utils_deepsdf.extract_mesh(grad_size_axis, sdf)
    except:
        print('Mesh extraction failed')
        return

    # save mesh as obj
    mesh_dir = os.path.join("./results/runs_sdf", cfg['folder_sdf'], 'meshes_training')
    if not os.path.exists(mesh_dir):
        os.mkdir(mesh_dir)
    obj_path = os.path.join(mesh_dir, f"mesh_{obj_idx}.obj")
    trimesh.exchange.export.export_mesh(trimesh.Trimesh(vertices, faces), obj_path, file_type='obj')


In [9]:
cfg_path = './config_files/reconstruct_from_latent.yaml'
with open(cfg_path, 'rb') as f:
    cfg = yaml.load(f, Loader=yaml.FullLoader)

In [16]:
# cfg['obj_ids'] = ['02942699/5d42d432ec71bfa1d5004b533b242ce6']
cfg['obj_ids'] = ['02942699/5d42d432ec71bfa1d5004b533b242ce6', '02876657/e9371d3abbb3bb7265bca0cae1ecfff5','03797390/ea127b5b9ba0696967699ff4ba91a25']

In [18]:
training_settings = read_params(cfg)

# Load the model
weights = os.path.join("./results/runs_sdf", cfg['folder_sdf'], 'weights.pt')

model = sdf_model.SDFModel(
    num_layers=training_settings['num_layers'],
    skip_connections=training_settings['latent_size'],
    latent_size=training_settings['latent_size'],
    inner_dim=training_settings['inner_dim']).to(device)
model.load_state_dict(torch.load(weights, map_location=device))

# Extract mesh obtained with the latent code optimised at inference
coords, grad_size_axis = utils_deepsdf.get_volume_coords(cfg['resolution'])
coords = coords.to(device)

# Split coords into batches because of memory limitations
coords_batches = torch.split(coords, 100000)

# Load paths
str2int_path = os.path.join("./results/runs_sdf", 'idx_str2int_dict.npy')
results_dict_path = os.path.join("./results/runs_sdf", cfg['folder_sdf'], 'results.npy')

# Load dictionaries
str2int_dict = np.load(str2int_path, allow_pickle=True).item()
results_dict = np.load(results_dict_path, allow_pickle=True).item()

for obj_id_path in cfg['obj_ids']:
    # Get object index in the results dictionary
    obj_idx = str2int_dict[obj_id_path]  # index in collected latent vector
    # Get the latent code optimised during training
    latent_code = results_dict['best_latent_codes'][obj_idx]
    latent_code = torch.tensor(latent_code).to(device)

    reconstruct_object(cfg, latent_code, obj_idx, model, coords_batches, grad_size_axis)



In [20]:
import trimesh

mesh = trimesh.load('./results/runs_sdf/17_07_172540/meshes_training/mesh_1.obj')

mesh.show()


In [21]:
latents = []

for obj_id_path in cfg['obj_ids']:
    # Get object index in the results dictionary
    obj_idx = str2int_dict[obj_id_path]  # index in collected latent vector
    # Get the latent code optimised during training
    latent_code = results_dict['best_latent_codes'][obj_idx]
    latent_code = torch.tensor(latent_code).to(device)
    latents.append(latent_code)

    # reconstruct_object(cfg, latent_code, obj_idx, model, coords_batches, grad_size_axis)





In [23]:
alpha = 0.5
linear_latent = alpha * latents[0] + (1 - alpha) * latents[2]
reconstruct_object(cfg, linear_latent, 4, model, coords_batches, grad_size_axis)

mesh = trimesh.load('./results/runs_sdf/17_07_172540/meshes_training/mesh_4.obj')

mesh.show()